# Exercice 4 : La revue des accès

> **En une phrase :** chaque compte doit appartenir à **une personne présente**, avec **juste** les droits dont elle a besoin, et une authentification forte.

**Les demandes de l'auditeur traitées ici :** D-05 (revue des droits d'accès, **critique**). Références : ISO 27001 A.5.18 et A.8.2, CIS Controls v8 5 et 6.

**Le scénario.** On croise trois sources : la liste des employés des **RH**, l'export des comptes **Microsoft 365**
(Entra ID), et les comptes locaux des serveurs (`/etc/passwd`, `/etc/group`). L'auditeur veut chaque anomalie, et l'action prévue.

**Le plan.**
1. Croiser les comptes Microsoft 365 avec les RH.
2. Quatre règles d'anomalie.
3. À vous : les comptes des serveurs.
4. Déposer la pièce.

## Étape 0 : exécutez cette cellule en premier
En local, ouvrez **le dossier du chapitre** dans VS Code. Sur Colab, la cellule télécharge le dossier. Elle doit afficher **Prêt : True**.

In [1]:
# Étape 0 : sur Colab, récupère le dossier du chapitre depuis GitHub. En local (VS Code), cette cellule ne fait rien.
import os
if not os.path.exists('auditeur.py'):
    !test -d /content/cours || git clone -q --depth 1 https://github.com/elyesmanai/Code-8INF857---2026.git /content/cours
    %cd "/content/cours/Chapitre 8 - Cadres réglementaires"
print('Prêt :', os.path.exists('auditeur.py'))

Prêt : True


## Étape 1 : Croiser les comptes avec les RH
**Ce qu'on fait :** on rattache chaque compte Microsoft 365 à un employé, grâce à l'identifiant (la partie avant `@`).

In [2]:
from helper import lire_csv, lire_serveur, deposer
import pandas as pd

rh = lire_csv('rh_employes.csv')
m365 = lire_csv('comptes_m365.csv')
m365['identifiant'] = m365['utilisateur'].str.split('@').str[0]
comptes = m365.merge(rh, on='identifiant', how='left')
print(f"{len(rh)} personnes aux RH ({(rh.statut == 'parti').sum()} départs), {len(m365)} comptes Microsoft 365")
comptes[['utilisateur', 'compte_actif', 'mfa', 'roles_admin', 'derniere_connexion', 'service', 'statut', 'date_depart']].head(10)

39 personnes aux RH (4 départs), 41 comptes Microsoft 365


,utilisateur,compte_actif,mfa,roles_admin,derniere_connexion,service,statut,date_depart
0,mgagnon@nordheb.ca,oui,oui,Global Administrator,2026-11-26,TI,actif,
1,jtremblay@nordheb.ca,oui,oui,,2026-11-26,RH,actif,
2,lcote@nordheb.ca,oui,oui,Exchange Administrator,2026-11-21,Finances,actif,
3,shaddad@nordheb.ca,oui,oui,,2026-11-26,Dév.,actif,
4,nbergeron@nordheb.ca,oui,non,Global Administrator,2026-11-20,Direction,actif,
5,pbouchard@nordheb.ca,oui,oui,,2026-09-14,TI,parti,2026-08-29
6,aroy@nordheb.ca,oui,non,,2026-11-24,Dév.,actif,
7,klavoie@nordheb.ca,non,oui,,2026-06-12,Soutien,parti,2026-06-12
8,sgirard@nordheb.ca,oui,non,,2026-10-01,Ventes,parti,2026-10-02
9,esimard@nordheb.ca,oui,oui,,2026-11-22,Soutien,actif,


## Étape 2 : Quatre règles d'anomalie

| Anomalie | La règle | Pourquoi c'est grave |
|---|---|---|
| `parti_actif` | compte actif d'une personne partie | un ancien employé (ou celui qui a volé son mot de passe) garde l'accès |
| `admin_injustifie` | rôle d'administrateur hors du service TI | moindre privilège : un compte compromis devient catastrophique |
| `sans_mfa` | compte actif d'un employé sans MFA | un mot de passe volé suffit pour entrer |
| `sans_proprietaire` | compte qui ne correspond à aucune personne | compte partagé ou de service : personne n'en répond |

**Ce qu'on fait :** on applique les quatre règles aux comptes **actifs**.

In [3]:
actifs = comptes[comptes['compte_actif'] == 'oui']
anomalies = []
def anomalie(compte, systeme, type_, detail):
    anomalies.append({'compte': compte, 'systeme': systeme, 'anomalie': type_, 'detail': detail})

for _, c in actifs.iterrows():
    if pd.isna(c['statut']):
        anomalie(c['identifiant'], 'm365', 'sans_proprietaire', f"aucun employé « {c['nom_affiche']} » aux RH")
    elif c['statut'] == 'parti':
        anomalie(c['identifiant'], 'm365', 'parti_actif', f"parti le {c['date_depart']}, dernière connexion le {c['derniere_connexion']}")
    else:
        if c['roles_admin'] and c['service'] != 'TI':
            anomalie(c['identifiant'], 'm365', 'admin_injustifie', f"{c['roles_admin']} ({c['poste']}, {c['service']})")
        if c['mfa'] != 'oui':
            anomalie(c['identifiant'], 'm365', 'sans_mfa', f"{c['poste']}" + (' — ADMINISTRATRICE' if c['roles_admin'] else ''))

pd.DataFrame(anomalies)

,compte,systeme,anomalie,detail
0,lcote,m365,admin_injustifie,"Exchange Administrator (Contrôleur, Finances)"
1,nbergeron,m365,admin_injustifie,"Global Administrator (Présidente, Direction)"
2,nbergeron,m365,sans_mfa,Présidente — ADMINISTRATRICE
3,pbouchard,m365,parti_actif,"parti le 2026-08-29, dernière connexion le 202..."
4,aroy,m365,sans_mfa,Développeuse
5,sgirard,m365,parti_actif,"parti le 2026-10-02, dernière connexion le 202..."
6,hlarouche,m365,parti_actif,"parti le 2026-08-21, dernière connexion le 202..."
7,svc-sauvegarde,m365,sans_proprietaire,aucun employé « Service sauvegarde » aux RH
8,accueil,m365,sans_proprietaire,aucun employé « Accueil (partagé) » aux RH


**👀 Ce qu'on doit voir :** 9 anomalies dans Microsoft 365 :
- **3 comptes de personnes parties** encore actifs (`pbouchard`, `sgirard`, `hlarouche`) ;
- **2 rôles d'administrateur hors TI** : la présidente est *Global Administrator*, le contrôleur *Exchange Administrator* ;
- **2 employés sans MFA**, dont la présidente, qui est aussi administratrice globale : le compte le plus dangereux de l'entreprise ;
- **2 comptes sans propriétaire** : `svc-sauvegarde` et `accueil` (partagé).

**❓ Question pour la classe :** regardez `pbouchard`. Il est parti le 29 août. Que signifie sa dernière connexion du **14 septembre** ?

*Piste : quelqu'un s'est connecté avec son compte **après** son départ. Ce n'est plus une anomalie d'accès, c'est un **incident**. On le retrouvera dans le registre des incidents (ex6, ticket INC-2026-041).*

## À vous : les comptes des serveurs
Les serveurs ont leurs propres comptes : `/etc/passwd` (un compte par ligne `nom:x:uid:...`) et `/etc/group` (la ligne `sudo:x:27:a,b,c` liste les administrateurs).
Les comptes **humains** ont un `uid` ≥ 1000 ; en dessous, ce sont des comptes système (`root`, `www-data`, `postgres`…).

Complétez `comptes_serveur(serveur)` : pour chaque compte avec `uid` ≥ 1000, appelez `anomalie(...)` avec le **nom du serveur** comme système :
- pas aux RH → `sans_proprietaire` ;
- parti → `parti_actif` ;
- actif, dans le groupe `sudo` et hors TI → `admin_injustifie`.

In [4]:
def comptes_serveur(serveur):
    passwd = lire_serveur(serveur, 'etc/passwd')
    groupes = lire_serveur(serveur, 'etc/group')
    # À VOUS
    pass

for serveur in ['web01', 'bd01']:
    comptes_serveur(serveur)
pd.DataFrame(anomalies).query("systeme != 'm365'")

,compte,systeme,anomalie,detail


*Réponse :*
```python
def comptes_serveur(serveur):
    passwd = lire_serveur(serveur, 'etc/passwd')
    groupes = lire_serveur(serveur, 'etc/group')
    sudo = next(l for l in groupes.splitlines() if l.startswith('sudo:')).split(':')[3].split(',')
    employes = rh.set_index('identifiant')
    for l in passwd.splitlines():
        nom, _, uid = l.split(':')[:3]
        if int(uid) < 1000:
            continue
        if nom not in employes.index:
            anomalie(nom, serveur, 'sans_proprietaire', 'compte local sans employé correspondant')
        elif employes.loc[nom, 'statut'] == 'parti':
            anomalie(nom, serveur, 'parti_actif', f"parti le {employes.loc[nom, 'date_depart']}")
        elif nom in sudo and employes.loc[nom, 'service'] != 'TI':
            anomalie(nom, serveur, 'admin_injustifie', f"sudo sur {serveur} ({employes.loc[nom, 'poste']})")
```
Attendu : `deploy` sur web01 (compte de déploiement sans propriétaire), `pbouchard` sur bd01 (parti, et toujours dans `sudo` !), `aroy` sur bd01 (développeuse avec `sudo` sur la base de production).

## Étape 3 : Une action pour chaque anomalie, et le dépôt
**Ce qu'on fait :** l'auditeur veut savoir ce qu'on **fait**. Une action type par anomalie.

In [5]:
ACTIONS = {
    'parti_actif': "Désactiver immédiatement ; relier le processus de départ RH à la désactivation des comptes",
    'admin_injustifie': "Retirer le rôle ; compte admin séparé au besoin, justifié et revu chaque trimestre",
    'sans_mfa': "Activer la MFA (obligatoire par accès conditionnel)",
    'sans_proprietaire': "Nommer un propriétaire responsable, ou supprimer le compte",
}
revue = pd.DataFrame(anomalies)
revue['action'] = revue['anomalie'].map(ACTIONS)
deposer('revue_acces.csv', revue)
revue[['compte', 'systeme', 'anomalie', 'action']]

Pièce déposée : livrables/revue_acces.csv


,compte,systeme,anomalie,action
0,lcote,m365,admin_injustifie,Retirer le rôle ; compte admin séparé au besoi...
1,nbergeron,m365,admin_injustifie,Retirer le rôle ; compte admin séparé au besoi...
2,nbergeron,m365,sans_mfa,Activer la MFA (obligatoire par accès conditio...
3,pbouchard,m365,parti_actif,Désactiver immédiatement ; relier le processus...
4,aroy,m365,sans_mfa,Activer la MFA (obligatoire par accès conditio...
5,sgirard,m365,parti_actif,Désactiver immédiatement ; relier le processus...
6,hlarouche,m365,parti_actif,Désactiver immédiatement ; relier le processus...
7,svc-sauvegarde,m365,sans_proprietaire,"Nommer un propriétaire responsable, ou supprim..."
8,accueil,m365,sans_proprietaire,"Nommer un propriétaire responsable, ou supprim..."


## Ce que dit l'auditeur
**Ce qu'on fait :** on relance l'auditeur automatique sur tout le dossier `livrables/`. D-05 est **critique**.

In [6]:
import auditeur
auditeur.auditer();

Cabinet Boréal Audit — revue des pièces déposées

⚠️ D-01 Textes applicables et justification          9.3 / 10
      · nis2 : applicabilité « ? » incorrecte
      · nis2 : justification manquante ou trop courte
⚠️ D-02 Inventaire des actifs rapproché du réseau   14.2 / 15  (critique)
      · 10.0.1.47 : aucun propriétaire
      · 10.0.3.58 : aucun propriétaire
✅ D-03 Configuration sécurisée des serveurs        15.0 / 15
❌ D-04 Sauvegardes et test de restauration          0.0 / 10  (critique)
      · web01 / sauvegarde_24h : absent
      · web01 / test_restauration_12_mois : absent
      · bd01 / sauvegarde_24h : absent
      · bd01 / test_restauration_12_mois : absent
⚠️ D-05 Revue des droits d'accès                    11.2 / 15  (critique)
      · anomalie non trouvée : aroy sur bd01 (admin_injustifie)
      · anomalie non trouvée : deploy sur web01 (sans_proprietaire)
      · anomalie non trouvée : pbouchard sur bd01 (parti_actif)
❌ D-06 Registre et plan de traitement des risques   

**👀 Ce qu'on doit voir :** D-05 à ⚠️ **11,2 / 15** sans votre tâche : les 3 anomalies des serveurs manquent. Avec la tâche, ✅ **15 / 15**.
Score : **50 / 100** sans les tâches, **65 / 100** avec les tâches des ex. 1 à 4.

## Ce qu'il faut retenir
- Une revue d'accès **croise des sources** : RH (qui est là ?), annuaire (quels comptes ?), systèmes (quels droits ?). Aucune ne suffit seule.
- Les quatre questions : la personne est-elle **encore là** ? A-t-elle **besoin** de ces droits ? A-t-elle la **MFA** ? **Qui répond** de ce compte ?
- Le problème n° 1 est presque toujours le **départ** mal fermé. Ici, il a mené à un vrai incident.
- Chaque anomalie vient avec une **action** : la revue sert à corriger, pas seulement à constater.

**Exercice suivant :** on a trouvé beaucoup de problèmes. Avec 50 000 $, lesquels traiter d'abord ? Le registre de risques.